In [1]:
# import pandas as pd
# import numpy as np
# from sklearn.preprocessing import StandardScaler
# from sklearn.cluster import DBSCAN
# from scipy.spatial.distance import cdist
# import os

# # ۱. تنظیمات مسیرها و متغیرها
# file_path = r'second_stage_inputs\G11\dsas_g11_bearings_vibration_temp_output.xlsx'
# output_filename1 = r'outputs\G11\dsas_g11_bearings_vibration_temp_correlation_detection\correlation_detection\dsas_g11_bearings_vibration_temp_correlation_detection_output1.xlsx'
# output_filename2 = r'outputs\G11\dsas_g11_bearings_vibration_temp_correlation_detection\correlation_detection\dsas_g11_bearings_vibration_temp_correlation_detection_output2.xlsx'

# os.makedirs(os.path.dirname(output_filename1), exist_ok=True)

# all_features = ['AssetID_9358', 'AssetID_9359', 'AssetID_9360', 'AssetID_9361', 
#                 'AssetID_9368', 'AssetID_9369', 'AssetID_9370', 'AssetID_12208',
#                 'AssetID_9343', 'AssetID_9344', 'AssetID_9408']

# target_sensors = ['AssetID_9358', 'AssetID_9359', 'AssetID_9360', 'AssetID_9361']

# # ۲. خواندن فایل اصلی
# try:
#     df_raw = pd.read_excel(file_path)
#     df_raw['date'] = pd.to_datetime(df_raw['date'])
#     df_raw = df_raw.sort_values(by='date')
#     print("فایل ورودی با موفقیت خوانده شد.")
# except Exception as e:
#     print(f"خطا در خواندن فایل: {e}")
#     exit()

# # ۳. پیش‌پردازش و حذف ۱۰ درصد داده‌های پرت (DBSCAN)
# scaler = StandardScaler()
# scaled_data = scaler.fit_transform(df_raw[all_features])

# dbscan = DBSCAN(eps=0.5, min_samples=5)
# labels = dbscan.fit_predict(scaled_data)

# cluster_centers = {cid: scaled_data[labels == cid].mean(axis=0) for cid in set(labels) if cid != -1}

# def calculate_distance(i):
#     label, point = labels[i], scaled_data[i].reshape(1, -1)
#     if label != -1:
#         return cdist(point, cluster_centers[label].reshape(1, -1))[0][0]
#     return np.min(cdist(point, np.array(list(cluster_centers.values())))) if cluster_centers else 0.0

# df_raw['distance'] = [calculate_distance(i) for i in range(len(df_raw))]
# df_cleaned = df_raw.sort_values(by='distance', ascending=False).iloc[int(len(df_raw)*0.1):].copy()
# df_cleaned = df_cleaned.sort_values(by='date').set_index('date')

# # ۴. بازه ۳۰ روز آخر
# last_date = df_cleaned.index.max()
# start_analysis_date = last_date - pd.Timedelta(days=30)
# print(f"شروع تحلیل Rolling از تاریخ: {start_analysis_date}")

# # ۵. محاسبه Rolling Correlation و ذخیره در لیست‌ها
# results_list = [] # برای خروجی شماره ۱

# print("در حال محاسبه ماتریس کوریلیشن متحرک...")

# # محاسبه کوریلیشن برای تمامی جفت‌ها به صورت یکجا برای بهینه‌سازی
# for target in target_sensors:
#     # ایجاد یک دیکشنری برای ذخیره مقادیر این تارگت در هر لحظه (برای خروجی ۲)
#     # این دیکشنری در نهایت به دیتافریم تبدیل می‌شود
#     temp_storage = {}

#     for feature in all_features:
#         if target == feature:
#             # کوریلیشن با خودش را NaN در نظر می‌گیریم
#             temp_storage[feature] = np.nan
#             continue

#         # محاسبه Rolling Correlation
#         rolling_series = df_cleaned[target].rolling(window='30D').corr(df_cleaned[feature])
#         # فقط بازه ۳۰ روز آخر را نگه می‌داریم
#         temp_storage[feature] = rolling_series[rolling_series.index >= start_analysis_date]


#     # ساخت ردیف‌های خروجی ۲ برای این تارگت خاص
#     target_df_wide = pd.DataFrame(temp_storage)
#     target_df_wide['AssetID'] = target
#     target_df_wide = target_df_wide.reset_index()

#     # مرتب‌سازی ستون‌ها: date, AssetID و سپس بقیه
#     cols = ['date', 'AssetID'] + all_features
#     results_list.append(target_df_wide[cols])

# # ۶. ترکیب نتایج و ساخت فایل‌های خروجی
# df_output2 = pd.concat(results_list, ignore_index=True)

# # تولید خروجی شماره ۱ از روی خروجی شماره ۲ (تبدیل Wide به Long)
# # این کار باعث می‌شود محاسبات دوباره تکرار نشود و هر دو فایل با هم منطبق باشند
# df_output1 = df_output2.melt(id_vars=['date', 'AssetID'], 
#                              value_vars=all_features, 
#                              var_name='AssetID_correlation', 
#                              value_name='correlation_value')

# # حذف مقادیر خودش با خودش (NaN) در خروجی ۱
# df_output1 = df_output1.dropna(subset=['correlation_value'])

# # ۷. ذخیره نهایی
# try:
#     # ذخیره خروجی ۱ (ساختار ردیفی)
#     df_output1.to_excel(output_filename1, index=False)
#     # ذخیره خروجی ۲ (ساختار ستونی - درخواستی جدید)
#     df_output2.to_excel(output_filename2, index=False)

#     print("\nعملیات با موفقیت پایان یافت.")
#     print(f"فایل ۱ (ساختار ردیفی): {output_filename1}")
#     print(f"فایل ۲ (ساختار ستونی): {output_filename2}")
#     print(f"تعداد ردیف‌های خروجی نهایی: {len(df_output2)}")
# except Exception as e:
#     print(f"خطا در ذخیره فایل‌ها: {e}")

# # نمایش نمونه خروجی ۲
# print("\nنمونه ساختار فایل شماره ۲:")
# print(df_output2.head())

In [ ]:
# import pandas as pd
# import numpy as np
# from sklearn.preprocessing import StandardScaler
# from sklearn.cluster import DBSCAN
# from scipy.spatial.distance import cdist
# import os

# # ۱. تنظیمات مسیرها و متغیرها
# file_path = r'second_stage_inputs\G11\dsas_g11_bearings_vibration_temp_output.xlsx'
# output_filename1 = r'outputs\G11\dsas_g11_bearings_vibration_temp_correlation_detection\correlation_detection\dsas_g11_bearings_vibration_temp_correlation_detection_output1.xlsx'
# output_filename2 = r'outputs\G11\dsas_g11_bearings_vibration_temp_correlation_detection\correlation_detection\dsas_g11_bearings_vibration_temp_correlation_detection_output2.xlsx'

# os.makedirs(os.path.dirname(output_filename1), exist_ok=True)

# all_features = ['AssetID_9358', 'AssetID_9359', 'AssetID_9360', 'AssetID_9361', 
#                 'AssetID_9368', 'AssetID_9369', 'AssetID_9370', 'AssetID_12208',
#                 'AssetID_9343', 'AssetID_9344', 'AssetID_9408']

# target_sensors = ['AssetID_9358', 'AssetID_9359', 'AssetID_9360', 'AssetID_9361']

# # سنسورهایی که قصد داریم مقادیر اصلی آن‌ها را در خروجی ۲ به صورت ستون جداگانه داشته باشیم
# additional_values = ['AssetID_9343', 'AssetID_9344']

# # ۲. خواندن فایل اصلی
# try:
#     df_raw = pd.read_excel(file_path)
#     df_raw['date'] = pd.to_datetime(df_raw['date'])
#     df_raw = df_raw.sort_values(by='date')
#     print("فایل ورودی با موفقیت خوانده شد.")
# except Exception as e:
#     print(f"خطا در خواندن فایل: {e}")
#     exit()

# # ۳. پیش‌پردازش و حذف ۱۰ درصد داده‌های پرت (DBSCAN)
# scaler = StandardScaler()
# scaled_data = scaler.fit_transform(df_raw[all_features])

# dbscan = DBSCAN(eps=0.5, min_samples=5)
# labels = dbscan.fit_predict(scaled_data)

# cluster_centers = {cid: scaled_data[labels == cid].mean(axis=0) for cid in set(labels) if cid != -1}

# def calculate_distance(i):
#     label, point = labels[i], scaled_data[i].reshape(1, -1)
#     if label != -1:
#         return cdist(point, cluster_centers[label].reshape(1, -1))[0][0]
#     return np.min(cdist(point, np.array(list(cluster_centers.values())))) if cluster_centers else 0.0

# df_raw['distance'] = [calculate_distance(i) for i in range(len(df_raw))]
# df_cleaned = df_raw.sort_values(by='distance', ascending=False).iloc[int(len(df_raw)*0.1):].copy()
# df_cleaned = df_cleaned.sort_values(by='date').set_index('date')

# # ۴. بازه ۳۰ روز آخر
# last_date = df_cleaned.index.max()
# start_analysis_date = last_date - pd.Timedelta(days=30)
# print(f"شروع تحلیل Rolling از تاریخ: {start_analysis_date}")

# # ۵. محاسبه Rolling Correlation
# results_list = []
# print("در حال محاسبه ماتریس کوریلیشن متحرک...")

# for target in target_sensors:
#     temp_storage = {}
#     for feature in all_features:
#         if target == feature:
#             temp_storage[feature] = np.nan
#             continue

#         rolling_series = df_cleaned[target].rolling(window='30D').corr(df_cleaned[feature])
#         temp_storage[feature] = rolling_series[rolling_series.index >= start_analysis_date]

#     target_df_wide = pd.DataFrame(temp_storage)
#     target_df_wide['AssetID'] = target

#     target_df_wide = target_df_wide.reset_index()

#     # ساختار ستونی اولیه
#     cols = ['date', 'AssetID'] + all_features
#     results_list.append(target_df_wide[cols])

# # ۶. ترکیب نتایج و افزودن مقادیر سنسورهای درخواستی
# df_output2 = pd.concat(results_list, ignore_index=True)

# # استخراج مقادیر واقعی (Original) برای ادغام
# df_additional = df_cleaned[additional_values].reset_index()

# # به دلیل اینکه ستون‌های 9343 و 9344 در df_output2 نتایج کوریلیشن هستند، 
# # نام ستون‌های حاوی مقدار واقعی را تغییر می‌دهیم تا تداخل ایجاد نشود
# df_additional = df_additional.rename(columns={
#     'AssetID_9343': 'AssetID_9343_Value',
#     'AssetID_9344': 'AssetID_9344_Value'
# })

# # ادغام مقادیر واقعی با جدول کوریلیشن
# df_output2 = pd.merge(df_output2, df_additional, on='date', how='left')

# # لیست جدید ستون‌های مقدار واقعی برای استفاده در melt
# new_value_cols = ['AssetID_9343_Value', 'AssetID_9344_Value']

# # ۷. تولید خروجی شماره ۱ (تبدیل Wide به Long)
# # برای جلوگیری از خطا، ستون‌هایی که در id_vars هستند نباید در value_vars باشند
# df_output1 = df_output2.melt(id_vars=['date', 'AssetID'] + new_value_cols, 
#                              value_vars=all_features, 
#                              var_name='AssetID_correlation', 
#                              value_name='correlation_value')

# # حذف مقادیر خودش با خودش (NaN)
# df_output1 = df_output1.dropna(subset=['correlation_value'])

# # ۸. ذخیره نهایی
# try:
#     df_output1 = df_output1.sort_values(by=['date', 'AssetID'])
#     df_output2 = df_output2.sort_values(by=['date', 'AssetID'])

#     df_output1.to_excel(output_filename1, index=False)
#     df_output2.to_excel(output_filename2, index=False)

#     print("\nعملیات با موفقیت پایان یافت.")
#     print(f"فایل ۱ (Long): {output_filename1}")
#     print(f"فایل ۲ (Wide + Values): {output_filename2}")

#     # نمایش نمونه جهت تست
#     print("\nستون‌های نهایی فایل ۲:")
#     print(df_output2.columns.tolist())

# except Exception as e:
#     print(f"خطا در ذخیره فایل‌ها: {e}")

فایل ورودی با موفقیت خوانده شد.
شروع تحلیل Rolling از تاریخ: 2023-11-02 12:49:12
در حال محاسبه ماتریس کوریلیشن متحرک...

عملیات با موفقیت پایان یافت.
فایل ۱ (Long): outputs\G11\dsas_g11_bearings_vibration_temp_correlation_detection\correlation_detection\dsas_g11_bearings_vibration_temp_correlation_detection_output1.xlsx
فایل ۲ (Wide + Values): outputs\G11\dsas_g11_bearings_vibration_temp_correlation_detection\correlation_detection\dsas_g11_bearings_vibration_temp_correlation_detection_output2.xlsx

ستون‌های نهایی فایل ۲:
['date', 'AssetID', 'AssetID_9358', 'AssetID_9359', 'AssetID_9360', 'AssetID_9361', 'AssetID_9368', 'AssetID_9369', 'AssetID_9370', 'AssetID_12208', 'AssetID_9343', 'AssetID_9344', 'AssetID_9408', 'AssetID_9343_Value', 'AssetID_9344_Value']


In [5]:
# import pandas as pd
# import numpy as np
# from sklearn.preprocessing import StandardScaler
# from sklearn.cluster import DBSCAN
# from sklearn.linear_model import LinearRegression
# from scipy.spatial.distance import cdist
# import os

# # ۱. تنظیمات مسیرها و متغیرها
# file_path = r'second_stage_inputs\G11\dsas_g11_bearings_vibration_temp_output.xlsx'
# output_filename1 = r'outputs\G11\dsas_g11_bearings_vibration_temp_correlation_detection\correlation_detection\dsas_g11_bearings_vibration_temp_correlation_detection_output1.xlsx'
# output_filename2 = r'outputs\G11\dsas_g11_bearings_vibration_temp_correlation_detection\correlation_detection\dsas_g11_bearings_vibration_temp_correlation_detection_output2.xlsx'
# output_filename3 = r'outputs\G11\dsas_g11_bearings_vibration_temp_correlation_detection\correlation_detection\dsas_g11_bearings_vibration_temp_correlation_detection_output3.xlsx'

# os.makedirs(os.path.dirname(output_filename1), exist_ok=True)

# all_features = ['AssetID_9358', 'AssetID_9359', 'AssetID_9360', 'AssetID_9361', 
#                 'AssetID_9368', 'AssetID_9369', 'AssetID_9370', 'AssetID_12208',
#                 'AssetID_9343', 'AssetID_9344', 'AssetID_9408']

# target_sensors = ['AssetID_9358', 'AssetID_9359', 'AssetID_9360', 'AssetID_9361']
# additional_values = ['AssetID_9343', 'AssetID_9344']

# # ۲. خواندن فایل اصلی
# try:
#     df_raw = pd.read_excel(file_path)
#     df_raw['date'] = pd.to_datetime(df_raw['date'])
#     df_raw = df_raw.sort_values(by='date')
#     print("فایل ورودی با موفقیت خوانده شد.")
# except Exception as e:
#     print(f"خطا در خواندن فایل: {e}")
#     exit()

# # ۳. پیش‌پردازش و حذف ۱۰ درصد داده‌های پرت (DBSCAN)
# scaler = StandardScaler()
# scaled_data = scaler.fit_transform(df_raw[all_features])

# dbscan = DBSCAN(eps=0.5, min_samples=5)
# labels = dbscan.fit_predict(scaled_data)

# cluster_centers = {cid: scaled_data[labels == cid].mean(axis=0) for cid in set(labels) if cid != -1}

# def calculate_distance(i):
#     label, point = labels[i], scaled_data[i].reshape(1, -1)
#     if label != -1:
#         return cdist(point, cluster_centers[label].reshape(1, -1))[0][0]
#     return np.min(cdist(point, np.array(list(cluster_centers.values())))) if cluster_centers else 0.0

# df_raw['distance'] = [calculate_distance(i) for i in range(len(df_raw))]
# df_cleaned = df_raw.sort_values(by='distance', ascending=False).iloc[int(len(df_raw)*0.1):].copy()
# df_cleaned = df_cleaned.sort_values(by='date').set_index('date')

# # ۴. بازه ۳۰ روز آخر
# last_date = df_cleaned.index.max()
# start_analysis_date = last_date - pd.Timedelta(days=30)
# df_last_30_days = df_cleaned[df_cleaned.index >= start_analysis_date].copy()

# # ۵. محاسبات مربوط به فایل ۱ و ۲ (Correlation)
# results_list = []
# print("در حال محاسبه ماتریس کوریلیشن متحرک...")

# for target in target_sensors:
#     temp_storage = {}
#     for feature in all_features:
#         if target == feature:
#             temp_storage[feature] = np.nan
#             continue
#         rolling_series = df_cleaned[target].rolling(window='30D').corr(df_cleaned[feature])
#         temp_storage[feature] = rolling_series[rolling_series.index >= start_analysis_date]

#     target_df_wide = pd.DataFrame(temp_storage)
#     target_df_wide['AssetID'] = target
#     target_df_wide = target_df_wide.reset_index()
#     results_list.append(target_df_wide[['date', 'AssetID'] + all_features])

# df_output2 = pd.concat(results_list, ignore_index=True)

# df_additional = df_cleaned[additional_values].reset_index()
# df_additional_renamed = df_additional.rename(columns={col: f"{col}_Value" for col in additional_values})
# df_output2 = pd.merge(df_output2, df_additional_renamed, on='date', how='left')

# df_output1 = df_output2.melt(id_vars=['date', 'AssetID'] + [f"{c}_Value" for c in additional_values], 
#                              value_vars=all_features, var_name='AssetID_correlation', value_name='correlation_value').dropna(subset=['correlation_value'])

# # ۶. محاسبات مربوط به فایل ۳ (Regression)
# print("در حال محاسبه رگرسیون برای جفت سنسورهای تارگت...")
# regression_results = []

# # ایجاد جفت‌های یکتا از سنسورهای تارگت (مثلاً ۹۳۵۸ با ۹۳۵۹)
# import itertools
# target_pairs = list(itertools.combinations(target_sensors, 2))

# for s1, s2 in target_pairs:
#     # آماده‌سازی داده‌ها برای رگرسیون
#     X = df_last_30_days[[s1]].values
#     y = df_last_30_days[s2].values

#     # فیت کردن مدل رگرسیون خطی
#     model = LinearRegression()
#     model.fit(X, y)
#     y_pred = model.predict(X)

#     # ایجاد دیتافریم موقت برای این جفت
#     pair_name = f"{s1}_{s2}"
#     temp_reg_df = pd.DataFrame({
#         'date': df_last_30_days.index,
#         'Sensor_pair': pair_name,
#         'Regression_Value': y_pred,
#         'AssetID_Value_S1': df_last_30_days[s1].values, # مقدار واقعی سنسور اول
#         'AssetID_Value_S2': df_last_30_days[s2].values  # مقدار واقعی سنسور دوم
#     })
#     regression_results.append(temp_reg_df)

# df_output3 = pd.concat(regression_results, ignore_index=True)

# # ۷. ذخیره نهایی هر سه فایل
# try:
#     df_output1.sort_values(by=['date', 'AssetID']).to_excel(output_filename1, index=False)
#     df_output2.sort_values(by=['date', 'AssetID']).to_excel(output_filename2, index=False)
#     df_output3.sort_values(by=['date', 'Sensor_pair']).to_excel(output_filename3, index=False)

#     print("\nعملیات با موفقیت پایان یافت.")
#     print(f"فایل ۱: {output_filename1}")
#     print(f"فایل ۲: {output_filename2}")
#     print(f"فایل ۳ (Regression): {output_filename3}")

# except Exception as e:
#     print(f"خطا در ذخیره فایل‌ها: {e}")

In [ ]:
# import pandas as pd
# import numpy as np
# from sklearn.preprocessing import StandardScaler
# from sklearn.cluster import DBSCAN
# from sklearn.linear_model import LinearRegression
# from scipy.spatial.distance import cdist
# import itertools
# import os

# # ۱. تنظیمات مسیرها و متغیرها
# file_path = r'second_stage_inputs\G11\dsas_g11_bearings_vibration_temp_output.xlsx'
# output_filename1 = r'outputs\G11\dsas_g11_bearings_vibration_temp_correlation_detection\correlation_detection\dsas_g11_bearings_vibration_temp_correlation_detection_output1.xlsx'
# output_filename2 = r'outputs\G11\dsas_g11_bearings_vibration_temp_correlation_detection\correlation_detection\dsas_g11_bearings_vibration_temp_correlation_detection_output2.xlsx'
# output_filename3 = r'outputs\G11\dsas_g11_bearings_vibration_temp_correlation_detection\correlation_detection\dsas_g11_bearings_vibration_temp_correlation_detection_output3.xlsx'

# os.makedirs(os.path.dirname(output_filename1), exist_ok=True)

# all_features = ['AssetID_9358', 'AssetID_9359', 'AssetID_9360', 'AssetID_9361', 
#                 'AssetID_9368', 'AssetID_9369', 'AssetID_9370', 'AssetID_12208',
#                 'AssetID_9343', 'AssetID_9344', 'AssetID_9408']

# target_sensors = ['AssetID_9358', 'AssetID_9359', 'AssetID_9360', 'AssetID_9361']
# additional_values = ['AssetID_9343', 'AssetID_9344']

# # ۲. خواندن فایل اصلی
# try:
#     df_raw = pd.read_excel(file_path)
#     df_raw['date'] = pd.to_datetime(df_raw['date'])
#     df_raw = df_raw.sort_values(by='date')
#     print("فایل ورودی با موفقیت خوانده شد.")
# except Exception as e:
#     print(f"خطا در خواندن فایل: {e}")
#     exit()

# # ۳. پیش‌پردازش و حذف ۱۰ درصد داده‌های پرت (DBSCAN)
# scaler = StandardScaler()
# scaled_data = scaler.fit_transform(df_raw[all_features])

# dbscan = DBSCAN(eps=0.5, min_samples=5)
# labels = dbscan.fit_predict(scaled_data)

# cluster_centers = {cid: scaled_data[labels == cid].mean(axis=0) for cid in set(labels) if cid != -1}

# def calculate_distance(i):
#     label, point = labels[i], scaled_data[i].reshape(1, -1)
#     if label != -1:
#         return cdist(point, cluster_centers[label].reshape(1, -1))[0][0]
#     return np.min(cdist(point, np.array(list(cluster_centers.values())))) if cluster_centers else 0.0

# df_raw['distance'] = [calculate_distance(i) for i in range(len(df_raw))]

# # اصلاح خطای سینتکس در این بخش (اضافه شدن علامت )
# df_cleaned = df_raw.sort_values(by='distance', ascending=False).iloc[int(len(df_raw) * 0.1):].copy()
# df_cleaned = df_cleaned.sort_values(by='date').set_index('date')

# # ۴. بازه ۳۰ روز آخر برای تحلیل
# last_date = df_cleaned.index.max()
# start_analysis_date = last_date - pd.Timedelta(days=30)
# df_last_30_days = df_cleaned[df_cleaned.index >= start_analysis_date].copy()

# # ۵. محاسبات مربوط به فایل ۱ و ۲ (Correlation)
# results_list = []
# print("در حال محاسبه ماتریس کوریلیشن متحرک...")

# for target in target_sensors:
#     temp_storage = {}
#     for feature in all_features:
#         if target == feature:
#             temp_storage[feature] = np.nan
#             continue
#         rolling_series = df_cleaned[target].rolling(window='30D').corr(df_cleaned[feature])
#         temp_storage[feature] = rolling_series[rolling_series.index >= start_analysis_date]

#     target_df_wide = pd.DataFrame(temp_storage)
#     target_df_wide['AssetID'] = target

#     target_df_wide = target_df_wide.reset_index()
#     results_list.append(target_df_wide[['date', 'AssetID'] + all_features])

# df_output2 = pd.concat(results_list, ignore_index=True)
# df_additional = df_cleaned[additional_values].reset_index()
# df_additional_renamed = df_additional.rename(columns={col: f"{col}_Value" for col in additional_values})
# df_output2 = pd.merge(df_output2, df_additional_renamed, on='date', how='left')

# # تولید خروجی ردیفی (Long)
# df_output1 = df_output2.melt(id_vars=['date', 'AssetID'] + [f"{c}_Value" for c in additional_values], 
#                              value_vars=all_features, var_name='AssetID_correlation', value_name='correlation_value').dropna(subset=['correlation_value'])

# # ۶. محاسبات مربوط به فایل ۳ (Regression + Moving Average)
# print("در حال محاسبه رگرسیون و میانگین متحرک ۲۴ ساعته...")
# regression_results = []
# target_pairs = list(itertools.combinations(target_sensors, 2))

# for s1, s2 in target_pairs:
#     X = df_last_30_days[[s1]].values
#     y = df_last_30_days[s2].values

#     model = LinearRegression()
#     model.fit(X, y)
#     y_pred = model.predict(X)

#     pair_name = f"{s1}_{s2}"
#     temp_reg_df = pd.DataFrame({
#         'date': df_last_30_days.index,
#         'Sensor_pair': pair_name,
#         'Regression_Value': y_pred,
#         'AssetID_Value_S1': df_last_30_days[s1].values,
#         'AssetID_Value_S2': df_last_30_days[s2].values
#     })

#     # محاسبه میانگین متحرک ۲۴ ساعته خروجی رگرسیون برای همین جفت
#     temp_reg_df = temp_reg_df.set_index('date')
#     temp_reg_df['Regression_MA_24h'] = temp_reg_df['Regression_Value'].rolling(window='24h').mean()
#     temp_reg_df = temp_reg_df.reset_index()

#     regression_results.append(temp_reg_df)

# df_output3 = pd.concat(regression_results, ignore_index=True)

# # ۷. ذخیره نهایی هر سه فایل
# try:
#     df_output1.sort_values(by=['date', 'AssetID']).to_excel(output_filename1, index=False)
#     df_output2.sort_values(by=['date', 'AssetID']).to_excel(output_filename2, index=False)
#     df_output3.sort_values(by=['date', 'Sensor_pair']).to_excel(output_filename3, index=False)

#     print("\nعملیات با موفقیت پایان یافت.")
#     print(f"فایل ۱: {output_filename1}")
#     print(f"فایل ۲: {output_filename2}")
#     print(f"فایل ۳ (Regression + MA): {output_filename3}")

# except Exception as e:
#     print(f"خطا در ذخیره فایل‌ها: {e}")

فایل ورودی با موفقیت خوانده شد.
در حال محاسبه ماتریس کوریلیشن متحرک...
در حال محاسبه رگرسیون و میانگین متحرک ۲۴ ساعته...

عملیات با موفقیت پایان یافت.
فایل ۱: outputs\G11\dsas_g11_bearings_vibration_temp_correlation_detection\correlation_detection\dsas_g11_bearings_vibration_temp_correlation_detection_output1.xlsx
فایل ۲: outputs\G11\dsas_g11_bearings_vibration_temp_correlation_detection\correlation_detection\dsas_g11_bearings_vibration_temp_correlation_detection_output2.xlsx
فایل ۳ (Regression + MA): outputs\G11\dsas_g11_bearings_vibration_temp_correlation_detection\correlation_detection\dsas_g11_bearings_vibration_temp_correlation_detection_output3.xlsx


In [ ]:
import pandas as pd
import numpy as np
from sklearn.preprocessing import StandardScaler
from sklearn.cluster import DBSCAN
from sklearn.linear_model import LinearRegression
from scipy.spatial.distance import cdist
from scipy.stats import zscore
import itertools
import os
import warnings
warnings.filterwarnings('ignore')

# ۱. تنظیمات مسیرها و متغیرها
file_path = r'second_stage_inputs\G11\dsas_g11_bearings_vibration_temp_output.xlsx'
output_filename1 = r'outputs\G11\dsas_g11_bearings_vibration_temp_correlation_detection\correlation_detection\dsas_g11_bearings_vibration_temp_correlation_detection_output1.xlsx'
output_filename2 = r'outputs\G11\dsas_g11_bearings_vibration_temp_correlation_detection\correlation_detection\dsas_g11_bearings_vibration_temp_correlation_detection_output2.xlsx'
output_filename3 = r'outputs\G11\dsas_g11_bearings_vibration_temp_correlation_detection\correlation_detection\dsas_g11_bearings_vibration_temp_correlation_detection_output3.xlsx'
output_filename4 = r'outputs\G11\dsas_g11_bearings_vibration_temp_correlation_detection\correlation_detection\dsas_g11_bearings_vibration_temp_correlation_detection_output4.xlsx'

os.makedirs(os.path.dirname(output_filename1), exist_ok=True)

all_features = ['AssetID_9358', 'AssetID_9359', 'AssetID_9360', 'AssetID_9361', 
                'AssetID_9368', 'AssetID_9369', 'AssetID_9370', 'AssetID_12208',
                'AssetID_9343', 'AssetID_9344', 'AssetID_9408']

target_sensors = ['AssetID_9358', 'AssetID_9359', 'AssetID_9360', 'AssetID_9361']
additional_values = ['AssetID_9343', 'AssetID_9344']

# ۲. خواندن فایل اصلی
try:
    df_raw = pd.read_excel(file_path)
    df_raw['date'] = pd.to_datetime(df_raw['date'])
    df_raw = df_raw.sort_values(by='date')
    print("فایل ورودی با موفقیت خوانده شد.")
except Exception as e:
    print(f"خطا در خواندن فایل: {e}")
    exit()

# ۳. پیش‌پردازش و حذف ۱۰ درصد داده‌های پرت (DBSCAN)
scaler = StandardScaler()
scaled_data = scaler.fit_transform(df_raw[all_features])

dbscan = DBSCAN(eps=0.5, min_samples=5)
labels = dbscan.fit_predict(scaled_data)

cluster_centers = {cid: scaled_data[labels == cid].mean(axis=0) for cid in set(labels) if cid != -1}

def calculate_distance(i):
    label, point = labels[i], scaled_data[i].reshape(1, -1)
    if label != -1:
        return cdist(point, cluster_centers[label].reshape(1, -1))[0][0]
    return np.min(cdist(point, np.array(list(cluster_centers.values())))) if cluster_centers else 0.0

df_raw['distance'] = [calculate_distance(i) for i in range(len(df_raw))]

df_cleaned = df_raw.sort_values(by='distance', ascending=False).iloc[int(len(df_raw) * 0.1):].copy()
df_cleaned = df_cleaned.sort_values(by='date').set_index('date')

# ۴. تعریف بازه‌ها برای Partial Correlation RCA
last_date = df_cleaned.index.max()
start_analysis_date = last_date - pd.Timedelta(days=30)

# بازه نرمال (baseline): ۳۰ روز قبل از بازه خرابی (یعنی روزهای -۶۰ تا -۳۰)
baseline_end = start_analysis_date - pd.Timedelta(days=1)
baseline_start = baseline_end - pd.Timedelta(days=30)
df_baseline = df_cleaned[(df_cleaned.index >= baseline_start) & (df_cleaned.index <= baseline_end)].copy()

# بازه خرابی (fault): ۳۰ روز آخر
df_fault = df_cleaned[df_cleaned.index >= start_analysis_date].copy()

print(f"\nبازه نرمال (baseline): {baseline_start} تا {baseline_end} ({len(df_baseline)} رکورد)")
print(f"بازه خرابی (fault): {start_analysis_date} تا {last_date} ({len(df_fault)} رکورد)")

# ============================================================
# ۵. توابع محاسبه همبستگی جزئی (Partial Correlation)
# ============================================================

def partial_correlation_matrix(data, columns):
    """
    محاسبه ماتریس همبستگی جزئی به روش Precision Matrix (معکوس ماتریس همبستگی)
    
    Parameters:
    -----------
    data : DataFrame
        داده‌های شامل ستون‌های مورد نظر
    columns : list
        لیست نام ستون‌ها
    
    Returns:
    --------
    partial_corr : DataFrame
        ماتریس همبستگی جزئی
    """
    # محاسبه ماتریس همبستگی معمولی
    corr_matrix = data[columns].corr().values
    
    # افزودن یک مقدار کوچک به قطر برای جلوگیری از تکینگی ماتریس
    corr_matrix_regularized = corr_matrix + np.eye(corr_matrix.shape[0]) * 1e-6
    
    # محاسبه ماتریس معکوس (Precision Matrix)
    try:
        precision_matrix = np.linalg.inv(corr_matrix_regularized)
    except np.linalg.LinAlgError:
        # اگر ماتریس معکوس‌پذیر نبود، از pseudo-inverse استفاده کن
        precision_matrix = np.linalg.pinv(corr_matrix_regularized)
    
    # محاسبه همبستگی جزئی از روی Precision Matrix
    # partial_corr[i,j] = -P[i,j] / sqrt(P[i,i] * P[j,j])
    p = precision_matrix.copy()
    diag = np.sqrt(np.diag(p))
    
    # جلوگیری از تقسیم بر صفر
    diag[diag == 0] = 1e-6
    
    partial_corr = -p / np.outer(diag, diag)
    
    # مقدار قطر را 1 قرار بده
    np.fill_diagonal(partial_corr, 1.0)
    
    # محدود کردن به بازه [-1, 1]
    partial_corr = np.clip(partial_corr, -1, 1)
    
    # تبدیل به DataFrame
    partial_corr_df = pd.DataFrame(partial_corr, index=columns, columns=columns)
    
    return partial_corr_df


def calculate_deviation_score(data_baseline, data_fault, columns):
    """
    محاسبه نمره انحراف برای هر سنسور (Z-score نسبت به baseline)
    
    Returns:
    --------
    deviation_scores : dict
        دیکشنری شامل نمره انحراف برای هر سنسور
    """
    deviation_scores = {}
    
    for col in columns:
        baseline_mean = data_baseline[col].mean()
        baseline_std = data_baseline[col].std()
        
        if baseline_std == 0:
            baseline_std = 1e-6
            
        fault_mean = data_fault[col].mean()
        z_score = abs((fault_mean - baseline_mean) / baseline_std)
        deviation_scores[col] = z_score
    
    return deviation_scores


# ============================================================
# ۶. محاسبه همبستگی جزئی برای هر دو بازه
# ============================================================

print("\nمحاسبه ماتریس همبستگی جزئی برای بازه نرمال...")
partial_corr_baseline = partial_correlation_matrix(df_baseline, all_features)

print("محاسبه ماتریس همبستگی جزئی برای بازه خرابی...")
partial_corr_fault = partial_correlation_matrix(df_fault, all_features)

# محاسبه تغییرات (دلتا)
delta_partial_corr = partial_corr_fault - partial_corr_baseline

# ============================================================
# ۷. محاسبه نمره انحراف هر سنسور
# ============================================================
deviation_scores = calculate_deviation_score(df_baseline, df_fault, all_features)

# ============================================================
# ۸. محاسبه شاخص RCA برای هر سنسور
# ============================================================
"""
شاخص RCA ترکیبی از:
1. Strength Score: مجموع قدرمطلق همبستگی جزئی سنسور با بقیه در بازه خرابی
2. Change Score: مجموع قدرمطلق تغییرات همبستگی جزئی سنسور با بقیه
3. Deviation Score: نمره انحراف سنسور از baseline
"""

# Strength Score در بازه خرابی
strength_scores = {}
for col in all_features:
    # حذف قطر (خود سنسور با خودش)
    strengths = [abs(partial_corr_fault.loc[col, other]) for other in all_features if other != col]
    strength_scores[col] = np.mean(strengths) if strengths else 0

# Change Score: مجموع تغییرات همبستگی جزئی
change_scores = {}
for col in all_features:
    changes = [abs(delta_partial_corr.loc[col, other]) for other in all_features if other != col]
    change_scores[col] = np.mean(changes) if changes else 0

# ترکیب نهایی: RCA Score (نرمالایز شده بین 0 تا 1)
def normalize_scores(scores_dict):
    values = np.array(list(scores_dict.values()))
    if values.max() == values.min():
        return {k: 0.5 for k in scores_dict.keys()}
    return {k: (v - values.min()) / (values.max() - values.min()) for k, v in scores_dict.items()}

strength_norm = normalize_scores(strength_scores)
change_norm = normalize_scores(change_scores)
deviation_norm = normalize_scores(deviation_scores)

# وزندهی: می‌توانید وزن‌ها را تنظیم کنید
# در RCA سنسورهایی که انحراف و تغییر شبکه بالایی دارند، مهم‌ترند
rca_scores = {}
for col in all_features:
    # وزن پیشنهادی: انحراف 0.4، تغییر 0.4، قدرت شبکه 0.2
    rca_scores[col] = (0.4 * deviation_norm[col] + 
                       0.4 * change_norm[col] + 
                       0.2 * strength_norm[col])

# ============================================================
# ۹. ساخت خروجی شماره ۴ (Partial Correlation RCA Report)
# ============================================================

# ۹.۱ ماتریس‌های اصلی
matrix_baseline_df = partial_corr_baseline.copy()
matrix_baseline_df.index.name = 'Sensor'
matrix_baseline_df.columns.name = 'Sensor'

matrix_fault_df = partial_corr_fault.copy()
matrix_fault_df.index.name = 'Sensor'
matrix_fault_df.columns.name = 'Sensor'

matrix_delta_df = delta_partial_corr.copy()
matrix_delta_df.index.name = 'Sensor'
matrix_delta_df.columns.name = 'Sensor'

# ۹.۲ جدول RCA نهایی (برای هر سنسور)
rca_summary = pd.DataFrame({
    'Sensor': all_features,
    'Deviation_Score': [deviation_scores[col] for col in all_features],
    'Deviation_Normalized': [deviation_norm[col] for col in all_features],
    'Strength_Score_Fault': [strength_scores[col] for col in all_features],
    'Strength_Normalized': [strength_norm[col] for col in all_features],
    'Change_Score_Delta': [change_scores[col] for col in all_features],
    'Change_Normalized': [change_norm[col] for col in all_features],
    'RCA_Score': [rca_scores[col] for col in all_features],
    'Root_Cause_Suspicion': ['HIGH' if rca_scores[col] > 0.7 else 
                              'MEDIUM' if rca_scores[col] > 0.4 else 
                              'LOW' for col in all_features]
})

# مرتب‌سازی بر اساس RCA Score (بالاترین اولویت ریشه‌یابی)
rca_summary = rca_summary.sort_values('RCA_Score', ascending=False).reset_index(drop=True)

# ۹.۳ جدول تغییرات جفت سنسورها (برای تحلیل دقیق‌تر)
pairwise_changes = []
for i, s1 in enumerate(all_features):
    for j, s2 in enumerate(all_features):
        if i < j:  # فقط یک بار برای هر جفت
            pairwise_changes.append({
                'Sensor_1': s1,
                'Sensor_2': s2,
                'Partial_Corr_Baseline': partial_corr_baseline.loc[s1, s2],
                'Partial_Corr_Fault': partial_corr_fault.loc[s1, s2],
                'Delta_Change': delta_partial_corr.loc[s1, s2],
                'Abs_Change': abs(delta_partial_corr.loc[s1, s2])
            })

df_pairwise = pd.DataFrame(pairwise_changes)
df_pairwise = df_pairwise.sort_values('Abs_Change', ascending=False).reset_index(drop=True)

# ============================================================
# ۱۰. ذخیره فایل شماره ۴ (چند شیت)
# ============================================================
try:
    with pd.ExcelWriter(output_filename4, engine='openpyxl') as writer:
        # شیت ۱: RCA Summary (لیست سنسورها بر اساس اولویت ریشه‌یابی)
        rca_summary.to_excel(writer, sheet_name='RCA_Summary', index=False)
        
        # شیت ۲: Partial Correlation - Baseline
        matrix_baseline_df.to_excel(writer, sheet_name='Partial_Corr_Baseline')
        
        # شیت ۳: Partial Correlation - Fault Period
        matrix_fault_df.to_excel(writer, sheet_name='Partial_Corr_Fault')
        
        # شیت ۴: Delta Change (Fault - Baseline)
        matrix_delta_df.to_excel(writer, sheet_name='Delta_Change')
        
        # شیت ۵: Pairwise Changes (برای تحلیل دقیق جفت سنسورها)
        df_pairwise.to_excel(writer, sheet_name='Pairwise_Changes', index=False)
        
        # شیت ۶: تفسیر نتایج (توضیحات)
        interpretation = pd.DataFrame({
            'Metric': ['RCA_Score > 0.7', 'RCA_Score 0.4-0.7', 'RCA_Score < 0.4',
                       'Interpretation', 'Root Cause Candidates'],
            'Description': ['High Suspicion', 'Medium Suspicion', 'Low Suspicion',
                            f'Sensor with highest RCA_Score → Most likely Root Cause',
                            f'Top candidate(s): {rca_summary.head(2)["Sensor"].tolist()}']
        })
        interpretation.to_excel(writer, sheet_name='Interpretation', index=False)
    
    print(f"\nفایل ۴ (Partial Correlation RCA): {output_filename4}")
    print("\n" + "="*70)
    print("🔍 نتایج تحلیل همبستگی جزئی برای RCA:")
    print("="*70)
    print("\n📊 رتبه‌بندی سنسورها بر اساس احتمال Root Cause:")
    print(rca_summary[['Sensor', 'RCA_Score', 'Root_Cause_Suspicion']].to_string(index=False))
    print("\n" + "="*70)
    print(f"🎯 سنسور(های) با بیشترین احتمال Root Cause: {rca_summary.head(2)['Sensor'].tolist()}")
    
except Exception as e:
    print(f"خطا در ذخیره فایل ۴: {e}")

# ادامه کد اصلی قبلی برای فایل‌های ۱، ۲، ۳
# ... (بقیه کد قبلی شما برای df_output1, df_output2, df_output3)

فایل ورودی با موفقیت خوانده شد.

بازه نرمال (baseline): 2023-10-02 12:49:12 تا 2023-11-01 12:49:12 (249 رکورد)
بازه خرابی (fault): 2023-11-02 12:49:12 تا 2023-12-02 12:49:12 (181 رکورد)

محاسبه ماتریس همبستگی جزئی برای بازه نرمال...
محاسبه ماتریس همبستگی جزئی برای بازه خرابی...

فایل ۴ (Partial Correlation RCA): outputs\G11\dsas_g11_bearings_vibration_temp_correlation_detection\correlation_detection\dsas_g11_bearings_vibration_temp_correlation_detection_output4_partial_corr_rca.xlsx

🔍 نتایج تحلیل همبستگی جزئی برای RCA:

📊 رتبه‌بندی سنسورها بر اساس احتمال Root Cause:
       Sensor  RCA_Score Root_Cause_Suspicion
AssetID_12208   0.752927                 HIGH
 AssetID_9368   0.705308                 HIGH
 AssetID_9358   0.699271               MEDIUM
 AssetID_9369   0.633476               MEDIUM
 AssetID_9343   0.476515               MEDIUM
 AssetID_9359   0.432985               MEDIUM
 AssetID_9361   0.414128               MEDIUM
 AssetID_9370   0.368790                  LOW
 AssetID_936